# Build Dual-Embedding FAISS Index (Raw Text + Profile)

Notebook này xây dựng **dual FAISS index** cho RAG pipeline mới:

```
vector = normalise( α·embed(raw_text) + (1−α)·embed(profile_text) )
```

**Yêu cầu trước khi chạy:**
- `data/profile_db/essays/profile_store.jsonl` đã được build bởi profiler (rag/profiler/runner.py)
- `models/sbert_essays_finetuned/` hoặc dùng base nomic nếu chưa có finetuned model

**Output:** `data/vector_db/essays_dual/vectors.faiss` + `vectors_meta.jsonl`

**Anti-leakage:** Test set không được embed vào index này.

## 1. CONFIG — Sửa các path ở đây

In [1]:
import os, sys
from pathlib import Path

# ════════════════════════════════════════════════════════════════════════════
# PROJECT ROOT — thư mục gốc chứa rag/, ptd_model/, data/, ...
# ════════════════════════════════════════════════════════════════════════════
PROJECT_ROOT = str(Path(os.getcwd()).parent.parent)  # notebook/gpt/ -> root
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# ════════════════════════════════════════════════════════════════════════════
# INPUT PATHS
# ════════════════════════════════════════════════════════════════════════════
TRAIN_CSV          = os.path.join(PROJECT_ROOT, "data", "split", "essays", "train.csv")
PROFILE_STORE_PATH = os.path.join(PROJECT_ROOT, "data", "profile_db", "essays", "profile_store.jsonl")

# ════════════════════════════════════════════════════════════════════════════
# MODEL
# Ưu tiên dùng finetuned model. Nếu chưa có, đặt về base nomic.
# ════════════════════════════════════════════════════════════════════════════
FINETUNED_MODEL_DIR = os.path.join(PROJECT_ROOT, "models", "none_finetuned")  
BASE_MODEL          = "nomic-ai/nomic-embed-text-v1.5"

# max_seq_length: 2048 covers 99.8% of essays without truncation.
# NomicBERT was trained with n_positions=2048, so no retraining needed.
MAX_SEQ_LENGTH = 2048

# ════════════════════════════════════════════════════════════════════════════
# DUAL-EMBEDDING CONFIG
# alpha: weight for raw-text embedding. (1-alpha) goes to profile.
# 0.5 = equal. Try 0.3–0.5 on val; lower = trust profile more.
# ════════════════════════════════════════════════════════════════════════════
ALPHA = 0.5

# ════════════════════════════════════════════════════════════════════════════
# OUTPUT
# ════════════════════════════════════════════════════════════════════════════
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "data", "vector_db", "essays_dual")

ENCODE_BATCH = 32

# ════════════════════════════════════════════════════════════════════════════
# Path check
# ════════════════════════════════════════════════════════════════════════════
print("=" * 60)
print("PATH CHECK")
print("=" * 60)
for label, p in [
    ("PROJECT_ROOT",       PROJECT_ROOT),
    ("TRAIN_CSV",          TRAIN_CSV),
    ("PROFILE_STORE_PATH", PROFILE_STORE_PATH),
]:
    ok = "OK" if Path(p).exists() else "MISSING"
    print(f"  {label:22s}: [{ok}]  {p}")

use_finetuned = Path(FINETUNED_MODEL_DIR).exists()
model_to_use  = FINETUNED_MODEL_DIR if use_finetuned else BASE_MODEL
print(f"\n  Embedding model    : {model_to_use}")
print(f"  Using finetuned    : {use_finetuned}")
print(f"  MAX_SEQ_LENGTH     : {MAX_SEQ_LENGTH}")
print(f"  ALPHA (raw weight) : {ALPHA}")
print(f"  OUTPUT_DIR         : {OUTPUT_DIR}")

PATH CHECK
  PROJECT_ROOT          : [OK]  f:\std\GR\code\model_x_ocean
  TRAIN_CSV             : [OK]  f:\std\GR\code\model_x_ocean\data\split\essays\train.csv
  PROFILE_STORE_PATH    : [OK]  f:\std\GR\code\model_x_ocean\data\profile_db\essays\profile_store.jsonl

  Embedding model    : nomic-ai/nomic-embed-text-v1.5
  Using finetuned    : False
  MAX_SEQ_LENGTH     : 2048
  ALPHA (raw weight) : 0.5
  OUTPUT_DIR         : f:\std\GR\code\model_x_ocean\data\vector_db\essays_dual


## 2. Imports

In [2]:
import json
import time
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer
from sklearn.metrics import accuracy_score, f1_score

# Project modules
import rag.embedder as _embedder
_embedder.FINETUNED_MODEL_DIR = FINETUNED_MODEL_DIR
_embedder.MAX_SEQ_LENGTH      = MAX_SEQ_LENGTH
_embedder.ALPHA               = ALPHA

from rag.profiler.store   import ProfileStore
from rag.profiler.prompts import FACETS
from rag.faiss_index      import FAISSIndex

print("Imports OK")

Imports OK


## 3. Load Data & Profile Store

In [3]:
train_df = pd.read_csv(TRAIN_CSV)
print(f"Train: {len(train_df)} essays")

store = ProfileStore(PROFILE_STORE_PATH)
store.load()
print(f"Profiles loaded: {len(store)}")

# Coverage check
valid_count   = sum(1 for i in range(len(train_df)) if store.has(f"user_{i}") and store.get(f"user_{i}").get("valid"))
invalid_count = len(train_df) - valid_count
print(f"  Valid profiles : {valid_count} / {len(train_df)} ({valid_count/len(train_df)*100:.1f}%)")
print(f"  Missing/invalid: {invalid_count}  (will fall back to raw-text embedding)")

# Label distribution
TRAITS = ["cOPN", "cCON", "cEXT", "cAGR", "cNEU"]
TRAIT_NAMES = {"cOPN": "Openness", "cCON": "Conscientiousness",
               "cEXT": "Extraversion", "cAGR": "Agreeableness", "cNEU": "Neuroticism"}
print("\nLabel distribution (train):")
for trait in TRAITS:
    col = train_df[trait].astype(str).str.lower()
    h = (col == "high").sum() or (col == "1").sum()
    l = (col == "low").sum()  or (col == "0").sum()
    print(f"  {TRAIT_NAMES[trait]:20s}: high={h:4d}, low={l:4d}")

Train: 1974 essays
Profiles loaded: 1974
  Valid profiles : 1974 / 1974 (100.0%)
  Missing/invalid: 0  (will fall back to raw-text embedding)

Label distribution (train):
  Openness            : high=1017, low= 957
  Conscientiousness   : high=1004, low= 970
  Extraversion        : high=1022, low= 952
  Agreeableness       : high=1046, low= 928
  Neuroticism         : high= 986, low= 988


## 4. Profile Serialisation Helper

In [4]:
def profile_to_full_text(profile):
    """Serialise all 30 facets + linguistic block to a flat string."""
    facets = profile.get("facets", {})
    ling   = profile.get("linguistic", {})
    lines  = []
    for code, _, _, _ in FACETS:
        f = facets.get(code)
        if f and f.get("signal"):
            ev = f.get("evidence", "")
            lines.append(f"{code} | {f['signal']} | {ev}")
    for key, val in ling.items():
        if val:
            lines.append(f"{key}: {val}")
    return "\n".join(lines)

# Sanity check: token length of a profile string
from rag.embedder import _get_model
model = _get_model()
tok   = model.tokenizer

sample_entry = store.get("user_0")
if sample_entry and sample_entry.get("valid"):
    sample_text = profile_to_full_text(sample_entry)
    n_tokens = len(tok.encode("search_document: " + sample_text, add_special_tokens=True, truncation=False))
    print(f"Sample profile token length: {n_tokens} (limit={MAX_SEQ_LENGTH})")
    print(f"Sample profile text (first 300 chars):\n{sample_text[:300]}")
else:
    print("user_0 not found or invalid — check profile store.")

[embedder] Loading embedding model: nomic-ai/nomic-embed-text-v1.5


modules.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/140 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

Sample profile token length: 633 (limit=2048)
Sample profile text (first 300 chars):
N1 | high | expresses nervousness about upcoming rowing practice and past injuries causing worry.
N2 | none | no evidence of anger or resentment toward others in the text.
N3 | mod | mentions feeling upset about missing a rowing meeting but does not express deep sadness.
N4 | mod | shows some nervou


## 5. Token Length Audit

In [5]:
EMBED_PREFIX = "search_document: "
raw_lengths  = []
prof_lengths = []

for i, row in train_df.iterrows():
    raw  = str(row["text"])
    ids  = tok.encode(EMBED_PREFIX + raw, add_special_tokens=True, truncation=False)
    raw_lengths.append(len(ids))

    entry = store.get(f"user_{i}")
    if entry and entry.get("valid"):
        pt   = profile_to_full_text(entry)
        pids = tok.encode(EMBED_PREFIX + pt, add_special_tokens=True, truncation=False)
        prof_lengths.append(len(pids))

raw_lengths  = np.array(raw_lengths)
prof_lengths = np.array(prof_lengths)

print(f"Raw text token lengths (n={len(raw_lengths)}):")
for pct in [50, 75, 90, 95, 99]:
    print(f"  p{pct:2d} = {int(np.percentile(raw_lengths, pct))}")
print(f"  max  = {raw_lengths.max()}")
print(f"  > {MAX_SEQ_LENGTH}: {(raw_lengths > MAX_SEQ_LENGTH).sum()} ({(raw_lengths > MAX_SEQ_LENGTH).mean()*100:.1f}%) — will use sliding window")

print(f"\nProfile token lengths (n={len(prof_lengths)}):")
print(f"  median={int(np.median(prof_lengths))}, max={prof_lengths.max()}  (always < {MAX_SEQ_LENGTH})")

Raw text token lengths (n=1974):
  p50 = 769
  p75 = 988
  p90 = 1196
  p95 = 1340
  p99 = 1656
  max  = 3185
  > 2048: 5 (0.3%) — will use sliding window

Profile token lengths (n=1974):
  median=626, max=775  (always < 2048)


## 6. Build Dual-Embedding Index

In [6]:
from rag.runners.build_features import build_index

build_index(
    data              = train_df,
    profile_store_path= PROFILE_STORE_PATH,
    output_dir        = OUTPUT_DIR,
    alpha             = ALPHA,
    force_rebuild     = False,   # set True to rebuild even if index exists
)

print("\nIndex files:")
for fname in sorted(os.listdir(OUTPUT_DIR)):
    size_mb = os.path.getsize(os.path.join(OUTPUT_DIR, fname)) / 1e6
    print(f"  {fname:30s} {size_mb:>8.2f} MB")

[build_index] Loaded 1974 profiles from 'f:\\std\\GR\\code\\model_x_ocean\\data\\profile_db\\essays\\profile_store.jsonl'
[build_index] Embedding 1974 essays (alpha=0.5) ...
  200/1974  elapsed=581s  ETA=5152s
  400/1974  elapsed=1167s  ETA=4591s
  600/1974  elapsed=1740s  ETA=3985s
  800/1974  elapsed=2324s  ETA=3410s
  1000/1974  elapsed=2883s  ETA=2808s
  1200/1974  elapsed=3453s  ETA=2227s
  1400/1974  elapsed=4027s  ETA=1651s
  1600/1974  elapsed=4617s  ETA=1079s
  1800/1974  elapsed=5204s  ETA=503s
[build_index] Embedded 1974 essays in 5748.1s  (no-profile fallbacks: 0)
[build_index] Saved -> f:\std\GR\code\model_x_ocean\data\vector_db\essays_dual/  (1974 vectors, dim=768)

Index files:
  vectors.faiss                      6.06 MB
  vectors_meta.jsonl                22.09 MB


## 7. Sanity Check — Retrieval-based Label Accuracy

Proxy metric: encode val essays with dual embedding, retrieve top-K from train index, majority-vote label.

In [7]:
VAL_CSV          = os.path.join(PROJECT_ROOT, "data", "split", "essays", "val.csv")
VAL_PROFILE_PATH = os.path.join(PROJECT_ROOT, "data", "profile_db", "essays_test", "profile_store.jsonl")
TOP_K            = 5

val_df    = pd.read_csv(VAL_CSV)
val_store = ProfileStore(VAL_PROFILE_PATH)
val_store.load()
print(f"Val: {len(val_df)} essays | val profiles: {len(val_store)}")

# Encode val with dual embedding
from rag.embedder import get_dual_embedding, get_embedding

print("\nEncoding val essays...")
val_embs = []
for i, row in val_df.iterrows():
    raw   = str(row["text"])
    entry = val_store.get(f"user_{i}")
    if entry and entry.get("valid"):
        pt  = profile_to_full_text(entry)
        vec = get_dual_embedding(raw, pt, alpha=ALPHA)
    else:
        vec = get_embedding(raw)
    val_embs.append(vec)
    if (i + 1) % 50 == 0:
        print(f"  {i + 1}/{len(val_df)}")

val_embs = np.array(val_embs, dtype="float32")
print(f"Val embeddings shape: {val_embs.shape}")

Val: 247 essays | val profiles: 50

Encoding val essays...
  50/247
  100/247
  150/247
  200/247
Val embeddings shape: (247, 768)


In [9]:
# Load dual index and search
dual_index = faiss.read_index(os.path.join(OUTPUT_DIR, "vectors.faiss"))
scores, neighbor_idx = dual_index.search(val_embs, TOP_K)

# Load train meta for labels
import json as _json
train_meta = []
with open(os.path.join(OUTPUT_DIR, "vectors_meta.jsonl"), encoding="utf-8") as f:
    for line in f:
        train_meta.append(_json.loads(line))

TRAIT_FULL = {
    "cOPN": "Openness to Experience",
    "cCON": "Conscientiousness",
    "cEXT": "Extraversion",
    "cAGR": "Agreeableness",
    "cNEU": "Neuroticism",
}

print(f"=== Retrieval-based label accuracy (top-{TOP_K} majority vote) ===\n")
print(f"{'Trait':22s} {'Acc':>8s}  {'Macro-F1':>10s}")
print("-" * 44)
for trait_col, trait_full in TRAIT_FULL.items():
    val_labels = []
    preds      = []
    for i, row in val_df.iterrows():
        gt = str(row[trait_col]).strip().lower()
        if gt in ("1", "1.0"):  gt = "high"
        if gt in ("0", "0.0"):  gt = "low"
        val_labels.append(gt)

        neighbor_labels = [train_meta[idx]["trait_labels"].get(trait_full, "low")
                           for idx in neighbor_idx[i] if idx >= 0]
        pred = "high" if neighbor_labels.count("high") > neighbor_labels.count("low") else "low"
        preds.append(pred)

    acc = accuracy_score(val_labels, preds)
    f1  = f1_score(val_labels, preds, pos_label="high", average="macro")
    print(f"  {TRAIT_NAMES[trait_col]:20s}  {acc:>8.4f}  {f1:>10.4f}")

=== Retrieval-based label accuracy (top-5 majority vote) ===

Trait                       Acc    Macro-F1
--------------------------------------------
  Openness                0.5061      0.4982
  Conscientiousness       0.5385      0.5383
  Extraversion            0.5223      0.5209
  Agreeableness           0.6194      0.6107
  Neuroticism             0.5101      0.5100


c:\Users\ASUS\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1561: UserWarning: Note that pos_label (set to 'high') is ignored when average != 'binary' (got 'macro'). You may use labels=[pos_label] to specify a single positive class.
  warnings.warn(
c:\Users\ASUS\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1561: UserWarning: Note that pos_label (set to 'high') is ignored when average != 'binary' (got 'macro'). You may use labels=[pos_label] to specify a single positive class.
  warnings.warn(
c:\Users\ASUS\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1561: UserWarning: Note that pos_label (set to 'high') is ignored when average != 'binary' (got 'macro'). You may use labels=[pos_label] to specify a single positive class.
  warnings.warn(
c:\Users\ASUS\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:15

## 8. Alpha Sweep (optional)

Re-embed val with different alpha values to find the best fusion weight.

In [10]:
# NOTE: this cell only re-encodes the val queries, not the index.
# For a full alpha sweep you would need to rebuild the index for each alpha.
# This is a query-side approximation that gives directional signal.

ALPHA_SWEEP = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]

print(f"Alpha sweep (query-side only, index fixed at alpha={ALPHA})\n")
print(f"{'Alpha':>6s}  ", end="")
for t in TRAITS:
    print(f"{TRAIT_NAMES[t]:>16s}", end="")
print(f"  {'Avg Acc':>8s}")
print("-" * (8 + 16 * len(TRAITS) + 10))

for alpha_q in ALPHA_SWEEP:
    q_embs = []
    for i, row in val_df.iterrows():
        raw   = str(row["text"])
        entry = val_store.get(f"user_{i}")
        if entry and entry.get("valid"):
            pt  = profile_to_full_text(entry)
            vec = get_dual_embedding(raw, pt, alpha=alpha_q)
        else:
            vec = get_embedding(raw)
        q_embs.append(vec)

    q_embs = np.array(q_embs, dtype="float32")
    _, n_idx = dual_index.search(q_embs, TOP_K)

    accs = []
    print(f"  {alpha_q:.1f}   ", end="")
    for trait_col, trait_full in TRAIT_FULL.items():
        val_labels = []
        preds      = []
        for i, row in val_df.iterrows():
            gt = str(row[trait_col]).strip().lower()
            if gt in ("1", "1.0"): gt = "high"
            if gt in ("0", "0.0"): gt = "low"
            val_labels.append(gt)
            nb = [train_meta[idx]["trait_labels"].get(trait_full, "low")
                  for idx in n_idx[i] if idx >= 0]
            preds.append("high" if nb.count("high") > nb.count("low") else "low")
        acc = accuracy_score(val_labels, preds)
        accs.append(acc)
        print(f"{acc:>16.4f}", end="")
    print(f"  {np.mean(accs):>8.4f}")

Alpha sweep (query-side only, index fixed at alpha=0.5)

 Alpha          OpennessConscientiousness    Extraversion   Agreeableness     Neuroticism   Avg Acc
--------------------------------------------------------------------------------------------------
  0.2             0.4858          0.5506          0.5304          0.6154          0.5061    0.5377
  0.3             0.4777          0.5506          0.5263          0.6032          0.5101    0.5336
  0.4             0.5101          0.5628          0.5182          0.6073          0.5020    0.5401
  0.5             0.5061          0.5385          0.5223          0.6194          0.5101    0.5393
  0.6             0.5182          0.5223          0.5425          0.6113          0.5304    0.5449
  0.7             0.5304          0.5466          0.5385          0.6154          0.5223    0.5506
  0.8             0.5304          0.5466          0.5304          0.6154          0.5263    0.5498
